# 線形回帰

ここでは、アニーリングを用いた線形回帰について説明します。

参考文献: https://arxiv.org/abs/2008.02355

### 線形回帰の損失関数

$n$個の変数を持つデータ(ベクトル)を $\boldsymbol{x}$ とし、それに対して予測したい目的値を $y$ とします。

$y$ が各変数の重み付き和で予測できると仮定すると、次のような重みベクトル $\boldsymbol{w}$ を考えることができます。

$$
\boldsymbol{x}^T \cdot \boldsymbol{w} = y
$$

次に、$m$個のデータからなる学習データセット $X$ ($m \times n$ 行列)と、目的データ $Y$ ($m$次元ベクトル)があるとします。
線形回帰の目標は、$X$ に含まれるすべてのデータ $\boldsymbol{x}$ に対して対応する $y$ を予測する、単一の重みベクトル $\boldsymbol{w}$ を求めることです。

これは以下のような関数最小化問題になります。

$$
\min_{\boldsymbol{w}} E(\boldsymbol{w}) =  || X\boldsymbol{w}  - Y ||^2
$$

$E(\boldsymbol{w})$ を変形すると、

$$
\min_{\boldsymbol{w}} E(\boldsymbol{w}) = \boldsymbol{w}^T X^T X \boldsymbol{w} - 2\boldsymbol{w}^T X^T Y + Y^T Y
$$

重み $\boldsymbol{w}$ を量子ビットの測定結果にエンコードするために、$\boldsymbol{w} = P\hat{\boldsymbol{w}}$ とします。
$\hat{\boldsymbol{w}}$ は量子ビットの測定結果 $\{0, 1\}$ からなるベクトルです。
また、最小化に影響しない $Y^T Y$ は省略します。

$$
\min_{\boldsymbol{w}} E(\boldsymbol{w}) = 　\hat{\boldsymbol{w}}^T P^T  X^T X P\hat{\boldsymbol{w}} - 2\hat{\boldsymbol{w}}^T P^T X^T Y
$$

すると、アニーリングで解くことのできる一般的なQUBO問題に帰着できます。

$$
\min_{\boldsymbol{w}} E(\boldsymbol{w}) = 　\hat{\boldsymbol{w}}^T A \hat{\boldsymbol{w}} + \hat{\boldsymbol{w}}^T \boldsymbol{b}
$$

ここで

$$
A = P^T  X^T X P
$$

であり、

$$
\boldsymbol{b} = -2P^T X^T Y
$$

QUBO行列のサイズは、データセットに含まれるデータの個数には依存しないことに注意してください。
サイズは (データの変数の数) $\times$ (重みの値をエンコードするために使用する量子ビット数) です。

blueqatを使ってこれを実装してみましょう。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

最初のステップは、データセットを作成することです。
今回は、あらかじめ推定したい重みを設定しておきます。
データセット $X$ をランダムに生成し、重み付けとノイズを加えた値を目的データ $Y$ とします。

In [2]:
w = np.array([0.25, 0.75, 0.5])
X = np.random.rand(100, 3)

y = X @ w + np.random.normal(scale = 0.05, size = X.shape[0])

scikit-learnを使って古典的な線形回帰を行ってみましょう。
目的データ $Y$ にノイズを加えているため多少のずれはありますが、あらかじめ設定した重みを高い精度で推定できます。

In [3]:
from sklearn import linear_model

skmodel = linear_model.LinearRegression()
skmodel.fit(X, y)
w_sk = skmodel.coef_
print("Predicted weight:", w_sk)
print("True weight:", w)

Predicted weight: [0.22035128 0.74046476 0.50280137]
True weight: [0.25 0.75 0.5 ]


QAOAは、重み $\boldsymbol{w}$ の各値をいくつかの量子ビットにエンコードします。
どの値をエンコードするかは、上記の式の変換行列 $P$ で任意に設定できます。

ここでは、単純に1つの重みパラメータを2つの量子ビットにエンコードし、$[0, 0.25, 0.5, 0.75]$ の4つの値から予測します。
使用する量子ビット数は $(\text{エンコードに使う量子ビット数}) \times (\text{変数の数}) = 2 \times 3 = 6$ です。

In [4]:
K = 2 # bit number for weight
d = 3 # Number of features

p = [2 ** (-i-1) for i in range(K)]
I = np.eye(d)
P = np.kron(I, p)

In [5]:
A = np.dot(np.dot(P.T, X.T), np.dot(X, P))
b = -2 * np.dot(np.dot(P.T, X.T), y)
QUBO = A + np.diag(b)
QUBO = np.triu(QUBO + QUBO.T) - np.eye(QUBO.shape[0]) * QUBO

In [6]:
from blueqat.utils import Vqe, QaoaAnsatz, from_qubo

step = 2
hamiltonian = from_qubo(QUBO)

# `step` now sets the number of variational QAOA layers, each optimized by
# gradient descent (rather than a cheap fixed Trotter-schedule granularity as
# in the old SDK). step=2 on this 6-qubit problem still runs in a few seconds
# with the default settings, so no reduction is needed.
ansatz = QaoaAnsatz(hamiltonian, step)
result = Vqe(ansatz).run()
b = result.circuit.run(shots=10)
sample = b.most_common(1)[0][0]
print(sample)

/Users/yuichirominato/blueqatSDK/.claude/worktrees/determined-mahavira-bf713e/blueqat/utils.py:399: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:823.)
  total_matrix = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.int64, device=device), torch.empty(0, dtype=torch.complex128, device=device), (dim, dim))


110010


アニーリングの結果を重みパラメータの値に変換し、最初に設定した値と比較します。

In [7]:
w_qa = P @ [int(i) for i in list(sample)]

print("Predicted weight:", w_qa)
print("True weight:", w)

Predicted weight: [0.75 0.   0.5 ]
True weight: [0.25 0.75 0.5 ]


正しく予測できました。ここでは、エンコーディングが最初に設定した重み $\boldsymbol{w}$ の値に合わせて決められているため、正確に推定できています。

実際には、エンコーディングによる誤差が生じます。
エンコーディングの誤差を減らすには、より多くの量子ビットを用意する必要があります。

In [8]:
b

Counter({'110010': 2,
         '111101': 1,
         '010110': 1,
         '111100': 1,
         '110000': 1,
         '010101': 1,
         '010011': 1,
         '100101': 1,
         '111110': 1})